In [ ]:
#loading libraries
import pyfesom2 as pf
import matplotlib.pylab as plt
import xarray as xr
import numpy as np
import matplotlib.tri as mtri
import matplotlib.path as mpath
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import time
from pathlib import Path
from utide import solve
import pandas as pd
from pyproj import Geod
from pyproj import Transformer
from matplotlib.path import Path as MplPath
from matplotlib.patches import Patch
from scipy.spatial import cKDTree

In [ ]:
# loading the mesh
meshpath = '/albedo/work/projects/p_trr181/ebagaeva/work/ebagaeva/3IGW/Meshes/farc/'
alpha, beta, gamma=[0, 0, 0]
mesh=pf.load_mesh(meshpath, abg=[alpha, beta, gamma], usepickle = False, usejoblib=True)

In [ ]:
# loading the data
data_path= '/albedo/work/projects/p_trr181/ebagaeva/work/ebagaeva/3IGW/JRA55_2014/Tides_march_2try/'

u = pf.get_data(data_path,
                   "u", 
                   2014,
                   mesh,
                   how='ori',
                   compute=False, 
                  chunks={'nod2':10000, 'nz1':5})

print (u.shape)

u_top = u[:,0,:]
v = pf.get_data(data_path,
                   "v", 
                   2014,
                   mesh,
                   how='ori',
                   compute=False, 
                  chunks={'nod2':10000, 'nz1':5})

print (v.shape)
v_top = v[:,0,:]

In [ ]:
# ============================================================
# Arctic sectors
# ============================================================

sectors = [
    {
        "name": "East Siberian Seas",
        "abbr": "ESS",
        "lon_min": 80,
        "lon_max": 170,
        "lat_min": 65,
        "lat_max": 82,
        "wrap": False,
    },
    {
        "name": "Barents Sea",
        "abbr": "BS",
        "lon_min": 20,
        "lon_max": 80,
        "lat_min": 65,
        "lat_max": 82,
        "wrap": False,
    },
    {
        "name": "Atlantic Sector",
        "abbr": "AS",
        "lon_min": -40,
        "lon_max": 20,
        "lat_min": 65,
        "lat_max": 82,
        "wrap": False,
    },
    {
        "name": "Baffin Bay",
        "abbr": "BB",
        "lon_min": -80,
        "lon_max": -40,
        "lat_min": 65,
        "lat_max": 82,
        "wrap": False,
    },
    {
        "name": "Canadian Sector",
        "abbr": "CS",
        "lon_min": -140,
        "lon_max": -80,
        "lat_min": 65,
        "lat_max": 82,
        "wrap": False,
    },
    {
        "name": "Pacific Sector",
        "abbr": "PS",
        "lon_min": 170,
        "lon_max": -140,
        "lat_min": 65,
        "lat_max": 82,
        "wrap": True,
    },
    {
        "name": "Central Arctic Ocean",
        "abbr": "CAO",
        "lon_min": -180,
        "lon_max": 180,
        "lat_min": 82,
        "lat_max": 90,
        "wrap": False,
    },
]

In [ ]:
def sector_mask(data, sector):
    """
    Select points belonging to a given Arctic sector.

    Longitudes are internally converted to [-180, 180].
    """

    # Convert longitude from either 0...360 or -180...180
    # to a common -180...180 convention
    lon = ((data["lon"].values + 180) % 360) - 180

    lat = data["lat"].values

    # Latitude condition
    lat_mask = (
        (lat >= sector["lat_min"])
        & (lat < sector["lat_max"])
    )

    # Central Arctic Ocean
    if sector["abbr"] == "CAO":
        return (
            (lat >= sector["lat_min"])
            & (lat <= sector["lat_max"])
        )

    # Longitude condition
    if sector["wrap"]:

        # Pacific Sector crosses the dateline:
        # 170E -> 140W
        lon_mask = (
            (lon >= sector["lon_min"])
            | (lon < sector["lon_max"])
        )

    else:

        lon_mask = (
            (lon >= sector["lon_min"])
            & (lon < sector["lon_max"])
        )

    return lat_mask & lon_mask

In [ ]:
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from pyproj import Transformer


# ============================================================
# 1. Compute FESOM element centroids
# ============================================================

elem = mesh.elem.copy()

# Convert connectivity to zero-based indexing only if necessary
if elem.min() == 1:
    elem -= 1


# Element centroid latitude
elem_lat = mesh.y2[elem].mean(axis=1)


# Element mean water depth [m]
# mesh.topo is negative below sea level
elem_depth = -mesh.topo[elem].mean(axis=1)

print(
    "Element depth range:",
    np.nanmin(elem_depth),
    "to",
    np.nanmax(elem_depth),
    "m"
)


# ------------------------------------------------------------
# Circular longitude mean
# Robust for elements close to the dateline
# ------------------------------------------------------------

node_lon_rad = np.deg2rad(
    mesh.x2[elem]
)

elem_lon = np.rad2deg(
    np.arctan2(
        np.sin(node_lon_rad).mean(axis=1),
        np.cos(node_lon_rad).mean(axis=1),
    )
)


# Longitude versions
elem_lon_360 = elem_lon % 360.0
elem_lon_180 = (elem_lon + 180.0) % 360.0 - 180.0


# ============================================================
# 2. Keep Arctic elements north of 65°N
# ============================================================

arctic_mask = elem_lat >= 65.0

arctic_elem = np.flatnonzero(
    arctic_mask
)

lat_arctic = elem_lat[
    arctic_elem
]

lon_arctic = elem_lon_180[
    arctic_elem
]

depth_arctic = elem_depth[
    arctic_elem
]

print(
    "FESOM elements north of 65°N:",
    len(arctic_elem)
)


# ============================================================
# 3. Transform to projected coordinates
#
# EPSG:3413 = NSIDC North Polar Stereographic
# ============================================================

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3413",
    always_xy=True,
)

x_arctic, y_arctic = transformer.transform(
    lon_arctic,
    lat_arctic,
)

xy_arctic = np.column_stack(
    [
        x_arctic,
        y_arctic,
    ]
)


# ============================================================
# 4. Calculate distance from shallow-water regions
#
# We use TWO definitions:
#
# Standard sectors:
#     shallow water = depth < 500 m
#
# Barents Sea:
#     shallow water = depth < 300 m
# ============================================================


# ============================================================
# 4a. Distance from water shallower than 500 m
#     Used for all sectors except Barents Sea
# ============================================================

shallow_mask_500 = (
    depth_arctic < 500.0
)

xy_shallow_500 = xy_arctic[
    shallow_mask_500
]

print(
    "Number of Arctic elements shallower than 500 m:",
    len(xy_shallow_500)
)


tree_500 = cKDTree(
    xy_shallow_500
)

distance_to_shallow_500_m, _ = tree_500.query(
    xy_arctic,
    k=1,
)

distance_to_shallow_500_km = (
    distance_to_shallow_500_m
    / 1000.0
)


# ============================================================
# 4b. Distance from water shallower than 300 m
#     Used only for Barents Sea
# ============================================================

shallow_mask_300 = (
    depth_arctic < 300.0
)

xy_shallow_300 = xy_arctic[
    shallow_mask_300
]

print(
    "Number of Arctic elements shallower than 300 m:",
    len(xy_shallow_300)
)


tree_300 = cKDTree(
    xy_shallow_300
)

distance_to_shallow_300_m, _ = tree_300.query(
    xy_arctic,
    k=1,
)

distance_to_shallow_300_km = (
    distance_to_shallow_300_m
    / 1000.0
)


# ============================================================
# 5. Build candidate dataframe
# ============================================================

candidate_df = pd.DataFrame(
    {
        "elem": arctic_elem,

        "lat": lat_arctic,

        # Longitude 0...360
        "lon": elem_lon_360[
            arctic_elem
        ],

        # Longitude -180...180
        "lon180": lon_arctic,

        "depth": depth_arctic,

        "distance_to_shallow_500_km":
            distance_to_shallow_500_km,

        "distance_to_shallow_300_km":
            distance_to_shallow_300_km,

        "x": x_arctic,
        "y": y_arctic,
    }
)


# ============================================================
# 6. Selection settings
# ============================================================

# Standard sectors:
# depth >= 500 m
# >= 50 km from water shallower than 500 m

standard_min_depth = 500.0
standard_min_distance = 50.0


# Barents Sea:
# depth >= 300 m
# >= 30 km from water shallower than 300 m

bs_min_depth = 300.0
bs_min_distance = 30.0


# Preliminary spatial reduction
candidate_cell_size = 20_000.0   # 20 km


# Final number of locations in each sector
n_points_per_sector = 50


sector_order = [
    "SS",
    "BS",
    "AS",
    "BB",
    "CS",
    "PS",
    "CAO",
]


print("\nSelection criteria:")

print(
    "Standard sectors: "
    f"depth >= {standard_min_depth:.0f} m, "
    f"distance from <{standard_min_depth:.0f} m water "
    f">= {standard_min_distance:.0f} km"
)

print(
    "Barents Sea: "
    f"depth >= {bs_min_depth:.0f} m, "
    f"distance from <{bs_min_depth:.0f} m water "
    f">= {bs_min_distance:.0f} km"
)

print(
    "Candidate grid:",
    candidate_cell_size / 1000.0,
    "km"
)

print(
    "Points per sector:",
    n_points_per_sector
)


# ============================================================
# 7. Helper:
# Reduce dense FESOM mesh to one candidate per grid cell
# ============================================================

def reduce_to_grid(
    df,
    cell_size=20_000.0,
):

    df = df.copy()

    # Projected grid indices
    df["ix"] = np.floor(
        df["x"] / cell_size
    ).astype(int)

    df["iy"] = np.floor(
        df["y"] / cell_size
    ).astype(int)


    candidate_rows = []


    for (ix, iy), group in df.groupby(
        ["ix", "iy"],
        sort=True,
    ):

        # Centre of projected grid cell
        cx = (
            ix + 0.5
        ) * cell_size

        cy = (
            iy + 0.5
        ) * cell_size


        # Distance of FESOM element centroids
        # from grid-cell centre
        distance_squared = (
            (group["x"] - cx) ** 2
            +
            (group["y"] - cy) ** 2
        )


        # Keep FESOM element closest to grid-cell centre
        candidate_rows.append(
            group.loc[
                distance_squared.idxmin()
            ]
        )


    return (
        pd.DataFrame(candidate_rows)
        .sort_values(
            ["y", "x", "elem"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 8. Helper:
# Deterministic farthest-point sampling
# ============================================================

def farthest_point_sample(
    df,
    n_points,
):

    if len(df) < n_points:

        raise ValueError(
            f"Only {len(df)} candidates available; "
            f"cannot select {n_points} points."
        )


    xy = df[
        ["x", "y"]
    ].to_numpy()


    selected_indices = np.empty(
        n_points,
        dtype=int,
    )


    # --------------------------------------------------------
    # Deterministic first point:
    # candidate closest to centre of candidate cloud
    # --------------------------------------------------------

    centre = xy.mean(
        axis=0
    )

    selected_indices[0] = np.argmin(
        np.sum(
            (xy - centre) ** 2,
            axis=1,
        )
    )


    # --------------------------------------------------------
    # Distance from every candidate to first selected point
    # --------------------------------------------------------

    minimum_distance_squared = np.sum(
        (
            xy
            - xy[selected_indices[0]]
        ) ** 2,
        axis=1,
    )

    minimum_distance_squared[
        selected_indices[0]
    ] = -np.inf


    # --------------------------------------------------------
    # Farthest-point sampling
    # --------------------------------------------------------

    for k in range(
        1,
        n_points,
    ):

        next_index = np.argmax(
            minimum_distance_squared
        )

        selected_indices[k] = (
            next_index
        )


        new_distance_squared = np.sum(
            (
                xy
                - xy[next_index]
            ) ** 2,
            axis=1,
        )


        minimum_distance_squared = np.minimum(
            minimum_distance_squared,
            new_distance_squared,
        )


        minimum_distance_squared[
            selected_indices[:k + 1]
        ] = -np.inf


    return (
        df.iloc[selected_indices]
        .copy()
        .reset_index(drop=True)
    )


# ============================================================
# 9. Select points separately in every sector
#
# STANDARD SECTORS:
#     depth >= 500 m
#     >= 50 km from <500 m water
#
# BARENTS SEA:
#     depth >= 300 m
#     >= 30 km from <300 m water
# ============================================================

selected_sector_points = []


for sector in sectors:

    abbr = sector["abbr"]


    # --------------------------------------------------------
    # Get all FESOM elements belonging to this sector
    # --------------------------------------------------------

    mask = sector_mask(
        candidate_df,
        sector,
    )


    sector_df = (
        candidate_df.loc[mask]
        .copy()
        .reset_index(drop=True)
    )


    print(
        "\n========================================"
    )

    print(
        f"{abbr}: "
        f"{len(sector_df)} total elements"
    )


    # ========================================================
    # Barents Sea
    # ========================================================

    if abbr == "BS":

        eligible = sector_df[
            (
                sector_df["depth"]
                >= bs_min_depth
            )
            &
            (
                sector_df[
                    "distance_to_shallow_300_km"
                ]
                >= bs_min_distance
            )
        ].copy()


        print(
            f"{abbr}: "
            f"{len(eligible)} eligible elements "
            f"(depth >= {bs_min_depth:.0f} m; "
            f"distance from <{bs_min_depth:.0f} m water "
            f">= {bs_min_distance:.0f} km)"
        )


    # ========================================================
    # All other Arctic sectors
    # ========================================================

    else:

        eligible = sector_df[
            (
                sector_df["depth"]
                >= standard_min_depth
            )
            &
            (
                sector_df[
                    "distance_to_shallow_500_km"
                ]
                >= standard_min_distance
            )
        ].copy()


        print(
            f"{abbr}: "
            f"{len(eligible)} eligible elements "
            f"(depth >= {standard_min_depth:.0f} m; "
            f"distance from <{standard_min_depth:.0f} m water "
            f">= {standard_min_distance:.0f} km)"
        )


    # --------------------------------------------------------
    # Reduce dense mesh to one candidate per 20-km cell
    # --------------------------------------------------------

    sector_candidates = reduce_to_grid(
        eligible,
        cell_size=candidate_cell_size,
    )


    print(
        f"{abbr}: "
        f"{len(sector_candidates)} "
        "20-km candidate cells"
    )


    # --------------------------------------------------------
    # Check that enough candidates exist
    # --------------------------------------------------------

    if (
        len(sector_candidates)
        < n_points_per_sector
    ):

        raise ValueError(
            f"{abbr}: only "
            f"{len(sector_candidates)} "
            "candidate locations are available, "
            f"but {n_points_per_sector} "
            "were requested."
        )


    # --------------------------------------------------------
    # Farthest-point sampling
    # --------------------------------------------------------

    selected = farthest_point_sample(
        sector_candidates,
        n_points=n_points_per_sector,
    )


    selected["sector"] = abbr

    selected_sector_points.append(
        selected
    )


# ============================================================
# 10. Combine all sectors
# ============================================================

sample_points = pd.concat(
    selected_sector_points,
    ignore_index=True,
)


# ============================================================
# 11. Keep useful columns
# ============================================================

sample_points = sample_points[
    [
        "elem",
        "lat",
        "lon",
        "lon180",
        "depth",
        "distance_to_shallow_500_km",
        "distance_to_shallow_300_km",
        "x",
        "y",
        "sector",
    ]
]


# ============================================================
# 12. Stable ordering
# ============================================================

sample_points["sector"] = pd.Categorical(
    sample_points["sector"],
    categories=sector_order,
    ordered=True,
)


sample_points = (
    sample_points
    .sort_values(
        [
            "sector",
            "lat",
            "lon180",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 13. Diagnostics
# ============================================================

print(
    "\n========================================"
)

print(
    "FINAL SELECTION"
)

print(
    "========================================"
)


# ------------------------------------------------------------
# Total number of points
# ------------------------------------------------------------

print(
    "\nFinal number of points:"
)

print(
    len(sample_points)
)


# ------------------------------------------------------------
# Number of points per sector
# ------------------------------------------------------------

print(
    "\nPoints per sector:"
)

print(
    sample_points[
        "sector"
    ].value_counts(
        sort=False
    )
)


# ------------------------------------------------------------
# Depth range
# ------------------------------------------------------------

print(
    "\nDepth range per sector [m]:"
)

print(
    sample_points
    .groupby(
        "sector",
        observed=True,
    )["depth"]
    .agg(
        [
            "min",
            "median",
            "max",
        ]
    )
    .round(0)
)


# ------------------------------------------------------------
# Standard sectors:
# distance from <500 m water
# ------------------------------------------------------------

standard_points = sample_points[
    sample_points["sector"] != "BS"
]


print(
    "\nStandard sectors:"
    "\ndistance from <500 m water [km]:"
)


print(
    standard_points
    .groupby(
        "sector",
        observed=True,
    )[
        "distance_to_shallow_500_km"
    ]
    .agg(
        [
            "min",
            "median",
            "max",
        ]
    )
    .round(1)
)


# ------------------------------------------------------------
# Barents Sea:
# distance from <300 m water
# ------------------------------------------------------------

bs_points = sample_points[
    sample_points["sector"] == "BS"
]


print(
    "\nBarents Sea:"
    "\ndistance from <300 m water [km]:"
)


print(
    bs_points[
        "distance_to_shallow_300_km"
    ]
    .agg(
        [
            "min",
            "median",
            "max",
        ]
    )
    .round(1)
)


# ============================================================
# 14. Save
# ============================================================

output_file = (
    "fesom350_sector_centroids_"
    "standard500m50km_"
    "BS300m30km.csv"
)


sample_points.to_csv(
    output_file,
    index=False,
)


print(
    "\nSaved:",
    output_file
)


display(
    sample_points.head(20)
)

In [ ]:
fig = plt.figure(figsize=(4, 4))
ax = plt.axes(
    projection=ccrs.NorthPolarStereo()
)

ax.set_extent(
    [-180, 180, 65, 90],
    crs=ccrs.PlateCarree(),
)

ax.add_feature(
    cfeature.LAND,
    facecolor="lightgray",
    zorder=0,
)

ax.coastlines(
    resolution="50m",
    linewidth=0.7,
)

ax.gridlines(
    linewidth=0.4,
    linestyle="--",
    alpha=0.5,
)

ax.scatter(
    sample_points["lon180"],
    sample_points["lat"],
    s=22,
    edgecolor="black",
    linewidth=0.3,
    transform=ccrs.PlateCarree(),
    zorder=3,
)

ax.set_title(
    "FESOM2 locations selected for tidal ellipse comparison\n"
    "50 points per Arctic sector",
    fontsize=11
)

plt.tight_layout()
plt.show()

In [ ]:
#extracting 350 time series:
from dask.diagnostics import ProgressBar

elems = sample_points["elem"].to_numpy(dtype=int)
lats = sample_points["lat"].to_numpy()
lons = sample_points["lon"].to_numpy()

time_all = pd.DatetimeIndex(u_top.time.values)

march_mask = (
    (time_all >= pd.Timestamp("2014-03-01 00:00:00")) &
    (time_all <  pd.Timestamp("2014-04-01 00:00:00"))
)

march_idx = np.flatnonzero(march_mask)
time_fesom = time_all[march_idx]

print("Number of times:", len(time_fesom))
print("Number of points:", len(elems))

with ProgressBar():
    u_100 = (
        u_top
        .isel(time=march_idx, elem=elems)
        .compute()
        .values
    )

with ProgressBar():
    v_100 = (
        v_top
        .isel(time=march_idx, elem=elems)
        .compute()
        .values
    )

# Ensure dimensions are time × point
if u_100.shape[0] != len(time_fesom):
    u_100 = u_100.T
    v_100 = v_100.T

print("u shape:", u_100.shape)
print("v shape:", v_100.shape)

In [ ]:
# save the time series
np.savez_compressed(
    "fesom350_JRA55_uv.npz",
    time=time_fesom.values,
    elem=elems,
    lat=lats,
    lon=lons,
    u=u_100,
    v=v_100,
)

print("Saved to fesom350_JRA55_uv.npz")

In [ ]:
# loading it back:
data = np.load("fesom350_JRA55_uv.npz", allow_pickle=True)

time_fesom = pd.to_datetime(data["time"])

elems = data["elem"]
lats = data["lat"]
lons = data["lon"]

u_100 = data["u"]
v_100 = data["v"]

print(u_100.shape)
print(v_100.shape)

In [ ]:
# Fit vector tidal harmonics with UTide

from utide import solve
from tqdm.auto import tqdm

constituents = ["M2", "S2", "N2", "K2", "K1", "O1", "P1", "Q1"]

ellipse_rows = []
coef_list = []

for i in tqdm(
    range(len(elems)),
    desc="Fitting tidal ellipses",
    unit="point"
):
    ui = u_100[:, i]
    vi = v_100[:, i]

    valid = (
        np.isfinite(ui) &
        np.isfinite(vi) &
        ~pd.isna(time_fesom)
    )

    if valid.sum() < 24:
        continue

    coef = solve(
        time_fesom[valid],
        ui[valid],
        vi[valid],                 # vector analysis
        lat=lats[i],
        constit=constituents,
        method="ols",
        conf_int="none",
        trend=False,
        verbose=False,
    )

    coef_list.append(coef)

    for j, name in enumerate(coef.name):
        ellipse_rows.append({
            "point": i,
            "elem": elems[i],
            "lat": lats[i],
            "lon": lons[i],
            "constituent": str(name).strip().upper(),
            "major_ms": coef.Lsmaj[j],
            "minor_ms": coef.Lsmin[j],
            "inclination_deg": coef.theta[j],
            "phase_deg": coef.g[j],
        })

fesom_ellipses = pd.DataFrame(ellipse_rows)

print(fesom_ellipses.shape)
fesom_ellipses.head()

In [ ]:
# inspecting M2:
m2 = (
    fesom_ellipses[
        fesom_ellipses["constituent"] == "M2"
    ]
    .sort_values("point")
    .reset_index(drop=True)
)

print("Number of M2 ellipses:", len(m2))

display(
    m2[
        [
            "point",
            "lat",
            "lon",
            "major_ms",
            "minor_ms",
            "inclination_deg",
            "phase_deg",
        ]
    ].head()
)

In [ ]:
# loading TPXO data

from pathlib import Path
import pandas as pd

outfile = Path(
    "/albedo/work/projects/p_trr181/ebagaeva/software/OTPSnc/"
    "tpxo_fesom350_complete_march2014_8const"
)

rows = []
bad_lines = []

with outfile.open() as f:
    for line in f:
        p = line.split()

        if len(p) == 9:
            try:
                rows.append([
                    float(p[0]),
                    float(p[1]),
                    p[2],
                    p[3],
                    float(p[4]),
                    float(p[5]),
                    float(p[6]),
                    float(p[7]),
                    float(p[8]),
                ])
            except ValueError:
                bad_lines.append(line.strip())

        elif "Site is out of model grid OR land" in line:
            bad_lines.append(line.strip())

tpxo_data = pd.DataFrame(
    rows,
    columns=[
        "lat", "lon", "date", "clock",
        "U", "V", "u", "v", "depth"
    ],
)

tpxo_data["time"] = pd.to_datetime(
    tpxo_data["date"] + " " + tpxo_data["clock"],
    format="%m.%d.%Y %H:%M:%S",
)

tpxo_data["u_ms"] = tpxo_data["u"] / 100.0
tpxo_data["v_ms"] = tpxo_data["v"] / 100.0

print("Rows:", len(tpxo_data))
print(
    "Unique processed points:",
    tpxo_data[["lat", "lon"]].drop_duplicates().shape[0]
)
print("Unique times:", tpxo_data["time"].nunique())
print("Skipped records:", len(bad_lines))

In [ ]:
constituents = ["M2", "S2", "N2", "K2", "K1", "O1", "P1", "Q1"]

valid_tpxo_points = (
    tpxo_data[["lat", "lon"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

ellipse_rows_tpxo = []

for i, point in tqdm(
    valid_tpxo_points.iterrows(),
    total=len(valid_tpxo_points),
    desc="Fitting TPXO ellipses",
    unit="point",
):
    lat0 = point["lat"]
    lon0 = point["lon"]

    station = (
        tpxo_data[
            np.isclose(tpxo_data["lat"], lat0, atol=1e-6) &
            np.isclose(tpxo_data["lon"], lon0, atol=1e-6)
        ]
        .sort_values("time")
        .reset_index(drop=True)
    )

    time_tpxo = pd.DatetimeIndex(station["time"])
    u_tpxo = station["u_ms"].to_numpy()
    v_tpxo = station["v_ms"].to_numpy()

    valid = (
        np.isfinite(u_tpxo) &
        np.isfinite(v_tpxo) &
        ~pd.isna(time_tpxo)
    )

    coef = solve(
        time_tpxo[valid],
        u_tpxo[valid],
        v_tpxo[valid],
        lat=lat0,
        constit=constituents,
        method="ols",
        conf_int="none",
        trend=False,
        verbose=False,
    )

    for j, name in enumerate(coef.name):
        ellipse_rows_tpxo.append({
            "point": i,
            "lat": lat0,
            "lon": lon0,
            "constituent": str(name).strip().upper(),
            "major_ms": coef.Lsmaj[j],
            "minor_ms": coef.Lsmin[j],
            "inclination_deg": coef.theta[j],
            "phase_deg": coef.g[j],
        })

tpxo_ellipses = pd.DataFrame(ellipse_rows_tpxo)

m2_tpxo = (
    tpxo_ellipses[
        tpxo_ellipses["constituent"] == "M2"
    ]
    .sort_values("point")
    .reset_index(drop=True)
)

m2_tpxo["lon180"] = (
    (m2_tpxo["lon"] + 180.0) % 360.0
) - 180.0

print("TPXO M2 ellipses:", len(m2_tpxo))

In [ ]:
# Matching and plotting together

In [ ]:
# Ensure both tables use -180...180 longitude
m2_fesom = m2.copy()
m2_tpxo_plot = m2_tpxo.copy()

m2_fesom["lon180"] = ((m2_fesom["lon"] + 180.0) % 360.0) - 180.0
m2_tpxo_plot["lon180"] = ((m2_tpxo_plot["lon"] + 180.0) % 360.0) - 180.0

# Keep only FESOM points that have a corresponding valid TPXO point
matched_fesom_rows = []

for _, tp in m2_tpxo_plot.iterrows():

    dlat = m2_fesom["lat"] - tp["lat"]

    # Circular longitude difference
    dlon = (
        m2_fesom["lon180"] - tp["lon180"] + 180.0
    ) % 360.0 - 180.0

    distance2 = dlat**2 + dlon**2
    nearest = distance2.idxmin()

    matched_fesom_rows.append(m2_fesom.loc[nearest])

m2_fesom_valid = (
    pd.DataFrame(matched_fesom_rows)
    .reset_index(drop=True)
)

print("FESOM matched ellipses:", len(m2_fesom_valid))
print("TPXO ellipses:", len(m2_tpxo_plot))

In [ ]:
#check coordinate difference
dlat = np.abs(
    m2_fesom_valid["lat"].to_numpy()
    - m2_tpxo_plot["lat"].to_numpy()
)

dlon = (
    m2_fesom_valid["lon180"].to_numpy()
    - m2_tpxo_plot["lon180"].to_numpy()
    + 180.0
) % 360.0 - 180.0

print("Maximum latitude difference:", dlat.max())
print("Maximum longitude difference:", np.abs(dlon).max())

In [ ]:
def draw_ellipse_set(
    ax,
    data,
    map_crs,
    geo_crs,
    scale_m_per_ms,
    angle,
    color,
    linewidth,
    zorder,
):
    """Draw one collection of tidal ellipses."""

    if data.empty:
        return

    centers = map_crs.transform_points(
        geo_crs,
        data["lon180"].to_numpy(),
        data["lat"].to_numpy(),
    )

    x_center = centers[:, 0]
    y_center = centers[:, 1]

    for j, row in enumerate(data.itertuples(index=False)):

        a = row.major_ms * scale_m_per_ms
        b = abs(row.minor_ms) * scale_m_per_ms
        theta = np.deg2rad(row.inclination_deg)

        dx = (
            a * np.cos(angle) * np.cos(theta)
            - b * np.sin(angle) * np.sin(theta)
        )

        dy = (
            a * np.cos(angle) * np.sin(theta)
            + b * np.sin(angle) * np.cos(theta)
        )

        ax.plot(
            x_center[j] + dx,
            y_center[j] + dy,
            color=color,
            linewidth=linewidth,
            transform=map_crs,
            zorder=zorder,
        )

In [ ]:
from matplotlib.path import Path as MplPath

def set_sector_clip(ax, sector, map_crs, geo_crs):
    """
    Clip a Cartopy GeoAxes to one curved Arctic sector.
    """

    lons, lats = sector_boundary_lonlat(sector)

    projected = map_crs.transform_points(
        geo_crs,
        np.asarray(lons),
        np.asarray(lats),
    )

    vertices = projected[:, :2]

    codes = np.full(
        len(vertices),
        MplPath.LINETO,
        dtype=np.uint8,
    )

    codes[0] = MplPath.MOVETO
    codes[-1] = MplPath.CLOSEPOLY

    boundary_path = MplPath(vertices, codes)

    ax.set_boundary(
        boundary_path,
        transform=map_crs,
    )

    return boundary_path

In [ ]:
def sector_boundary_lonlat(sector, n=300):
    """
    Build a closed curved sector polygon in lon/lat coordinates.
    """

    lat_min = sector["lat_min"]
    lat_max = sector["lat_max"]

    # Central Arctic Ocean: full annulus/cap
    if sector["abbr"] == "CAO":
        lon_outer = np.linspace(-180, 180, n)
        lat_outer = np.full(n, lat_min)

        # The pole closes the sector
        lons = np.concatenate([
            lon_outer,
            [180, -180]
        ])

        lats = np.concatenate([
            lat_outer,
            [90, 90]
        ])

        return lons, lats

    lon1 = sector["lon_min"]
    lon2 = sector["lon_max"]

    # Unwrap a sector crossing the dateline
    if sector.get("wrap", False):
        if lon2 < lon1:
            lon2 += 360

    # Outer latitude arc
    outer_lon = np.linspace(lon1, lon2, n)
    outer_lat = np.full(n, lat_max)

    # Inner latitude arc, reversed
    inner_lon = np.linspace(lon2, lon1, n)
    inner_lat = np.full(n, lat_min)

    lons = np.concatenate([
        outer_lon,
        inner_lon,
        [outer_lon[0]]
    ])

    lats = np.concatenate([
        outer_lat,
        inner_lat,
        [outer_lat[0]]
    ])

    return lons, lats

In [ ]:
def plot_two_models_one_sector(
    ax,
    fesom_data,
    tpxo_data,
    sector,
    map_crs,
    geo_crs,
    scale_m_per_ms,
    angle,
    fesom_color="tab:blue",
    tpxo_color="tab:orange",
    ellipse_linewidth=1.8,
):
    """Plot FESOM and TPXO ellipses together in one sector."""

    fesom_subset = (
        fesom_data.loc[sector_mask(fesom_data, sector)]
        .copy()
        .reset_index(drop=True)
    )

    tpxo_subset = (
        tpxo_data.loc[sector_mask(tpxo_data, sector)]
        .copy()
        .reset_index(drop=True)
    )

    # Curved sector boundary and zoom
    set_sector_clip(ax, sector, map_crs, geo_crs)
    set_sector_extent(ax, sector, geo_crs)

    ax.add_feature(
        cfeature.LAND,
        facecolor="lightgray",
        zorder=0,
    )

    ax.coastlines(
        resolution="50m",
        linewidth=0.6,
        zorder=2,
    )

    ax.gridlines(
        linewidth=0.35,
        linestyle="--",
        alpha=0.45,
        zorder=1,
    )

    # Plot TPXO first and FESOM second
    draw_ellipse_set(
        ax=ax,
        data=tpxo_subset,
        map_crs=map_crs,
        geo_crs=geo_crs,
        scale_m_per_ms=scale_m_per_ms,
        angle=angle,
        color=tpxo_color,
        linewidth=ellipse_linewidth,
        zorder=3,
    )

    draw_ellipse_set(
        ax=ax,
        data=fesom_subset,
        map_crs=map_crs,
        geo_crs=geo_crs,
        scale_m_per_ms=scale_m_per_ms,
        angle=angle,
        color=fesom_color,
        linewidth=ellipse_linewidth,
        zorder=4,
    )

    ax.set_title(
        f"{sector['name']} ({sector['abbr']})",
        fontsize=14,
    )

    return fesom_subset, tpxo_subset

In [ ]:
def set_sector_extent(ax, sector, geo_crs, pad_deg=2.0):
    """
    Zoom the map to the geographic neighborhood of one sector.
    """

    if sector["abbr"] == "CAO":
        ax.set_extent(
            [-180, 180, sector["lat_min"] - 1, 90],
            crs=geo_crs,
        )
        return

    lon1 = sector["lon_min"]
    lon2 = sector["lon_max"]

    if sector.get("wrap", False) and lon2 < lon1:
        lon2 += 360

    ax.set_extent(
        [
            lon1 - pad_deg,
            lon2 + pad_deg,
            sector["lat_min"] - 1,
            sector["lat_max"] + 1,
        ],
        crs=geo_crs,
    )

In [ ]:
from matplotlib.lines import Line2D

geo_crs = ccrs.PlateCarree()
map_crs = ccrs.NorthPolarStereo(central_longitude=0)

# Identical scale for both models
scale_km_per_ms = 1500.0
scale_m_per_ms = scale_km_per_ms * 1000.0

angle = np.linspace(0, 2 * np.pi, 120)

fesom_color = "tab:blue"
tpxo_color = "tab:orange"
ellipse_linewidth = 1.8

fig, axes = plt.subplots(
    2,
    4,
    figsize=(15, 8),
    subplot_kw={"projection": map_crs},
)

axes = axes.ravel()

comparison_subsets = {}

for ax, sector in zip(axes[:7], sectors):

    fesom_subset, tpxo_subset = plot_two_models_one_sector(
        ax=ax,
        fesom_data=m2_fesom_valid,
        tpxo_data=m2_tpxo_plot,
        sector=sector,
        map_crs=map_crs,
        geo_crs=geo_crs,
        scale_m_per_ms=scale_m_per_ms,
        angle=angle,
        fesom_color=fesom_color,
        tpxo_color=tpxo_color,
        ellipse_linewidth=ellipse_linewidth,
    )

    comparison_subsets[sector["abbr"]] = {
        "fesom": fesom_subset,
        "tpxo": tpxo_subset,
    }

axes[7].set_visible(False)

legend_handles = [
    Line2D(
        [0], [0],
        color=fesom_color,
        linewidth=ellipse_linewidth,
        label="FESOM2",
    ),
    Line2D(
        [0], [0],
        color=tpxo_color,
        linewidth=ellipse_linewidth,
        label="TPXO",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, 0.01),
    fontsize=14,
    frameon=True,
)

fig.suptitle(
    "M2 surface-current tidal ellipses: FESOM2 and TPXO",
    fontsize=18,
    fontweight="bold",
)

plt.tight_layout(rect=[0, 0.07, 1, 0.95])

plt.savefig(
    "FESOM2_TPXO_M2_ellipses_by_Arctic_sector.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

# Number of plotted locations in each sector
for sector, data in comparison_subsets.items():
    print(
        f"{sector}: "
        f"FESOM2 = {len(data['fesom'])}, "
        f"TPXO = {len(data['tpxo'])}"
    )

In [ ]:
# improvement

In [ ]:
# 1. Number of ellipses by sector
for sector, data in comparison_subsets.items():
    print(
        f"{sector}: "
        f"FESOM2 = {len(data['fesom'])}, "
        f"TPXO = {len(data['tpxo'])}"
    )

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# 1. PUBLICATION SETTINGS
# ============================================================

FIG_WIDTH = 7.5
FIG_HEIGHT = 3.0

FS_LABEL = 9
FS_HEADER = 10
FS_SCALE = 7.5
FS_MODEL = 9
FS_LEGEND = 8

COLOR_INDIVIDUAL = "0.55"
ALPHA_INDIVIDUAL = 0.35

LW_INDIVIDUAL = 0.5
LW_REP = 2.0

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300,
})

# ============================================================
# 2. VELOCITY SETTINGS
# ============================================================

# Use mm/s throughout the figure.
# This permits integer limits even for weak tidal currents.

VELOCITY_FACTOR = 1000.0
VELOCITY_UNIT = r"mm s$^{-1}$"

# Representative ellipse should occupy approximately
# this fraction of the panel half-width.
REP_FRACTION = 0.55

# Percentile safeguard for individual ellipses.
INDIVIDUAL_PERCENTILE = 90
INDIVIDUAL_MARGIN = 1.05

# Minimum half-range [mm/s]
MIN_LIM = 1.0

# Optional manual limits in mm/s.
# Leave empty for automatic scaling.
MANUAL_LIMITS = {}

# Example:
# MANUAL_LIMITS = {
#     "ESS": 12,
#     "BS": 100,
#     "AS": 50,
#     "BB": 100,
#     "CS": 5,
#     "PS": 6,
#     "CAO": 10,
# }

# ============================================================
# 3. LAYOUT SETTINGS
# ============================================================

LEFT = 0.105
RIGHT = 0.985

GAP_X = 0.018
GAP_Y = 0.025

BOTTOM_ROW_Y = 0.27

# ============================================================
# 4. HELPER: TIDAL ELLIPSE COORDINATES
# ============================================================

def ellipse_xy(major, minor, inclination_deg, n=200):
    """
    Construct a centered tidal ellipse.

    Inputs:
        major, minor [m/s]
        inclination_deg [degrees]

    Returns:
        x, y [mm/s]
    """

    theta = np.linspace(0, 2 * np.pi, n)

    major = major * VELOCITY_FACTOR
    minor = minor * VELOCITY_FACTOR

    x0 = major * np.cos(theta)
    y0 = minor * np.sin(theta)

    inc = np.deg2rad(inclination_deg)

    x = (
        x0 * np.cos(inc)
        - y0 * np.sin(inc)
    )

    y = (
        x0 * np.sin(inc)
        + y0 * np.cos(inc)
    )

    return x, y


# ============================================================
# 5. HELPER: REPRESENTATIVE ORIENTATION
# ============================================================

def representative_orientation(angles_deg):

    angles = np.asarray(angles_deg, dtype=float)
    angles = angles[np.isfinite(angles)]

    if len(angles) == 0:
        return np.nan

    doubled = np.deg2rad(2 * angles)

    mean_angle = np.arctan2(
        np.mean(np.sin(doubled)),
        np.mean(np.cos(doubled))
    )

    return (np.rad2deg(mean_angle) / 2) % 180


# ============================================================
# 6. HELPER: REPRESENTATIVE ELLIPSE
# ============================================================

def representative_ellipse(df):

    major = np.nanmedian(df["major_ms"])

    minor_abs = np.nanmedian(
        np.abs(df["minor_ms"])
    )

    minor_sign = np.sign(
        np.nanmedian(df["minor_ms"])
    )

    if minor_sign == 0:
        minor_sign = 1

    minor = minor_sign * minor_abs

    inclination = representative_orientation(
        df["inclination_deg"].values
    )

    return major, minor, inclination


# ============================================================
# 7. INTEGER VELOCITY LIMIT
# ============================================================

def integer_limit(lim):
    """
    Round upward to the nearest integer.

    Examples:
        4.2  -> 5
        11.3 -> 12
        56.1 -> 57

    Unlike the previous version, this does not
    force limits to 1, 2, 5, 10, 20, etc.
    """

    if not np.isfinite(lim) or lim <= 0:
        return 1

    return max(1, int(np.ceil(lim)))


# ============================================================
# 8. SECTORS
# ============================================================

sector_order = [
    "ESS",
    "BS",
    "AS",
    "BB",
    "CS",
    "PS",
    "CAO"
]

# ============================================================
# 9. FIGURE GEOMETRY
# ============================================================

fig = plt.figure(
    figsize=(FIG_WIDTH, FIG_HEIGHT)
)

ncols = len(sector_order)

panel_w = (
    RIGHT - LEFT - (ncols - 1) * GAP_X
) / ncols

# Physically square panels
panel_h = panel_w * FIG_WIDTH / FIG_HEIGHT

top_row_y = BOTTOM_ROW_Y + panel_h + GAP_Y

if top_row_y + panel_h > 0.98:
    raise ValueError(
        "Panels exceed figure height. "
        "Reduce BOTTOM_ROW_Y or GAP_Y."
    )

axes = np.empty((2, ncols), dtype=object)

for col in range(ncols):

    x0 = LEFT + col * (panel_w + GAP_X)

    axes[0, col] = fig.add_axes([
        x0,
        top_row_y,
        panel_w,
        panel_h
    ])

    axes[1, col] = fig.add_axes([
        x0,
        BOTTOM_ROW_Y,
        panel_w,
        panel_h
    ])

# ============================================================
# 10. PLOT SECTORS
# ============================================================

sector_limits = {}

for col, abbr in enumerate(sector_order):

    fesom = comparison_subsets[abbr]["fesom"]
    tpxo = comparison_subsets[abbr]["tpxo"]

    # --------------------------------------------------------
    # Representative ellipses
    # --------------------------------------------------------

    major_f, _, _ = representative_ellipse(fesom)
    major_t, _, _ = representative_ellipse(tpxo)

    max_rep_major = max(
        abs(major_f),
        abs(major_t)
    ) * VELOCITY_FACTOR

    # --------------------------------------------------------
    # Automatic velocity limit
    # --------------------------------------------------------

    lim_rep = max_rep_major / REP_FRACTION

    all_major = np.concatenate([
        np.abs(fesom["major_ms"].to_numpy()),
        np.abs(tpxo["major_ms"].to_numpy())
    ])

    all_major = all_major[np.isfinite(all_major)]

    if len(all_major) > 0:

        lim_individual = (
            np.percentile(
                all_major,
                INDIVIDUAL_PERCENTILE
            )
            * VELOCITY_FACTOR
            * INDIVIDUAL_MARGIN
        )

    else:
        lim_individual = 0.0

    lim_raw = max(
        lim_rep,
        lim_individual,
        MIN_LIM
    )

    # --------------------------------------------------------
    # Final integer limit
    # --------------------------------------------------------

    if abbr in MANUAL_LIMITS:
        lim = MANUAL_LIMITS[abbr]
    else:
        lim = integer_limit(lim_raw)

    sector_limits[abbr] = lim

    # --------------------------------------------------------
    # Plot both datasets
    # --------------------------------------------------------

    for row, df in enumerate([fesom, tpxo]):

        ax = axes[row, col]

        # Individual ellipses
        for _, r in df.iterrows():

            values = [
                r["major_ms"],
                r["minor_ms"],
                r["inclination_deg"]
            ]

            if not np.all(np.isfinite(values)):
                continue

            x, y = ellipse_xy(
                r["major_ms"],
                r["minor_ms"],
                r["inclination_deg"]
            )

            ax.plot(
                x,
                y,
                color=COLOR_INDIVIDUAL,
                linewidth=LW_INDIVIDUAL,
                alpha=ALPHA_INDIVIDUAL,
                zorder=1
            )

        # Representative ellipse
        major_rep, minor_rep, inc_rep = (
            representative_ellipse(df)
        )

        if np.all(np.isfinite([
            major_rep,
            minor_rep,
            inc_rep
        ])):

            x_rep, y_rep = ellipse_xy(
                major_rep,
                minor_rep,
                inc_rep
            )

            ax.plot(
                x_rep,
                y_rep,
                color="black",
                linewidth=LW_REP,
                zorder=5
            )

        # Centre marker
        ax.plot(
            0,
            0,
            marker="+",
            color="black",
            markersize=3,
            markeredgewidth=0.7,
            zorder=6
        )

        # ----------------------------------------------------
        # Zero-reference lines
        # ----------------------------------------------------

        ax.axhline(
            0,
            color="0.82",
            linewidth=0.5,
            zorder=0
        )

        ax.axvline(
            0,
            color="0.82",
            linewidth=0.5,
            zorder=0
        )

        # ----------------------------------------------------
        # Shared sector limits
        # ----------------------------------------------------

        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)

        ax.set_aspect(
            "equal",
            adjustable="box"
        )

        # ----------------------------------------------------
        # Remove frames, ticks and grid
        # ----------------------------------------------------

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.set_xticks([])
        ax.set_yticks([])

        ax.grid(False)

    # --------------------------------------------------------
    # Sector title and velocity scale
    # --------------------------------------------------------

    x_center = (
        LEFT
        + col * (panel_w + GAP_X)
        + panel_w / 2
    )

    fig.text(
        x_center,
        top_row_y + panel_h + 0.075,
        abbr,
        ha="center",
        va="center",
        fontsize=FS_HEADER,
        fontweight="bold"
    )

    fig.text(
        x_center,
        top_row_y + panel_h + 0.025,
        rf"$\pm {lim}$ mm s$^{{-1}}$",
        ha="center",
        va="center",
        fontsize=FS_SCALE,
        color="0.35"
    )

# ============================================================
# 11. ROW LABELS
# ============================================================

for row, name in enumerate(["FESOM2", "TPXO"]):

    y_center = (
        top_row_y + panel_h / 2
        if row == 0
        else BOTTOM_ROW_Y + panel_h / 2
    )

    fig.text(
        0.055,
        y_center,
        name,
        rotation=90,
        ha="center",
        va="center",
        fontsize=FS_MODEL,
        fontweight="bold"
    )

# ============================================================
# 12. SHARED AXIS LABELS
# ============================================================

fig.text(
    0.50,
    0.17,
    VELOCITY_UNIT.join(["Zonal velocity (", ")"]),
    ha="center",
    va="center",
    fontsize=FS_LABEL
)

panels_y_center = (
    BOTTOM_ROW_Y + panel_h + GAP_Y / 2
)

fig.text(
    0.012,
    panels_y_center,
    VELOCITY_UNIT.join(["Meridional velocity (", ")"]),
    ha="center",
    va="center",
    rotation=90,
    fontsize=FS_LABEL
)

# ============================================================
# 13. LEGEND
# ============================================================

legend_handles = [

    Line2D(
        [0], [0],
        color=COLOR_INDIVIDUAL,
        linewidth=0.8,
        alpha=0.6,
        label="Individual locations"
    ),

    Line2D(
        [0], [0],
        color="black",
        linewidth=LW_REP,
        label="Representative ellipse"
    )
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.045),
    ncol=2,
    frameon=False,
    fontsize=FS_LEGEND,
    handlelength=2.5,
    columnspacing=2.5
)

# ============================================================
# 14. PRINT VELOCITY LIMITS
# ============================================================

print("\nVelocity limits by sector [mm/s]:")
print("--------------------------------")

for abbr in sector_order:
    print(f"{abbr:4s}: ±{sector_limits[abbr]}")

# ============================================================
# 15. SAVE
# ============================================================

output = "FESOM2_TPXO_M2_ellipses_horizontal_integer_mm"

fig.savefig(
    output + ".pdf",
    bbox_inches="tight",
    pad_inches=0.04
)

fig.savefig(
    output + ".png",
    dpi=400,
    bbox_inches="tight",
    pad_inches=0.04
)

plt.show()
